In [ ]:
# Week 3 Day 1 : Cross-Validation 

## Today's Goal

Today I wil learn how cros-validation gives a more reliable estimate of model performance than relying on a single validation spilt.

## Learning Objective

* Understand the limitation of a sigle validation split
* Understand the intuition behind K-Fold Cross-Validation
* Understand why Stratified K-Fold is useful for classification
* Use 'Cross_val_score' in scikit-learn
* Interpret cross-validation mean and standard deviation
* Explain why cross-validation gives a more stable performance estimate

## Expected Output

By the end of today, I should be able to:

1. Run cross-validation on a classification
2. Interpret the scores from different folds
3. Calculate and explain the mean CV score
4. Explain what the standard deviation tells me about model stability
5. Explain why crosss-validation is more reliable than one validation split

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Load the Banknote Authentication dataset
banknote = fetch_openml(
    data_id=1462,
    as_frame=True
)

X = banknote.data
y = banknote.target

print("Full X shape:", X.shape)
print("Full y shape:", y.shape)

# Hold out 20% of the data as the final test set
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nDevelopment set:", X_dev.shape)
print("Test set:", X_test.shape)

Full X shape: (1372, 4)
Full y shape: (1372,)

Development set: (1097, 4)
Test set: (275, 4)


In [6]:
## Inspecting Stratified K-Fold splits

from sklearn.model_selection import StratifiedKFold

# Create a 5-fold stratified cross-validation splitter
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_dev, y_dev),
    start=1
):
    y_train_fold = y_dev.iloc[train_idx]
    y_val_fold = y_dev.iloc[val_idx]

    print(f"Fold {fold}")
    print("Training samples:", len(train_idx))
    print("Validation samples:", len(val_idx))

    print("Validation class proportions:")
    print(y_val_fold.value_counts(normalize=True))

    print("-" * 40)

Fold 1
Training samples: 877
Validation samples: 220
Validation class proportions:
Class
1    0.554545
2    0.445455
Name: proportion, dtype: float64
----------------------------------------
Fold 2
Training samples: 877
Validation samples: 220
Validation class proportions:
Class
1    0.554545
2    0.445455
Name: proportion, dtype: float64
----------------------------------------
Fold 3
Training samples: 878
Validation samples: 219
Validation class proportions:
Class
1    0.552511
2    0.447489
Name: proportion, dtype: float64
----------------------------------------
Fold 4
Training samples: 878
Validation samples: 219
Validation class proportions:
Class
1    0.557078
2    0.442922
Name: proportion, dtype: float64
----------------------------------------
Fold 5
Training samples: 878
Validation samples: 219
Validation class proportions:
Class
1    0.557078
2    0.442922
Name: proportion, dtype: float64
----------------------------------------


In [8]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# Build the KNN pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

# Run 5-fold stratified cross-validation
cv_scores = cross_val_score(
    knn_pipeline,
    X_dev,
    y_dev,
    cv=skf,
    scoring="accuracy"
)

print("CV accuracy scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())
print("CV standard deviation:", cv_scores.std())

CV accuracy scores: [0.99545455 0.99545455 1.         1.         1.        ]
Mean CV accuracy: 0.9981818181818183
CV standard deviation: 0.002226808857075604


In [ ]:
# Reflection

1. Why is cross-validation more reliable than a single validation split?
Because cross-validations have K different validation folds and we can estimate by mean accuracy and Std. At the same time, a single 
validation split depends on a particular split.
2. What do the mean CV score and the CV standard deviation tell us?
Mean CV score shows average performance of the model and CV standard deviation tells us the variability across folds.
3. Why do we still need a final test set after using cross-validation?
Because a final test has an independent dataset and avoids overfitting.